# 🌐 NusaTranslation-Senti: Sentiment Classification Fine-Tuning
**Deep Learning | Hugging Face Transformers | Google Colab**

| Item | Detail |
|---|---|
| **Dataset** | `indonlp/nusatranslation_senti` (dari [nusa-writes](https://github.com/IndoNLP/nusa-writes)) |
| **Task** | Sentiment Classification (Positive / Neutral / Negative) |
| **Bahasa** | Jawa (`jav`), Minangkabau (`min`), Sunda (`sun`) |
| **Model** | IndoBERT · mBERT · XLM-RoBERTa |
| **Platform** | Google Colab (GPU T4 recommended) |
| **Paper** | NusaWrites — AACL 2023 |

> **Dataset config format:**  
> `nusatranslation_senti_{lang_code}_nusantara_text`  
> Kolom: `id`, `text`, `label` (string: `positive` / `neutral` / `negative`)

---


## 📦 Cell 1 — Install Libraries


In [ ]:
# Install semua dependensi yang diperlukan
# Jalankan sekali per runtime Colab

!pip install transformers datasets evaluate accelerate scikit-learn \
             pandas matplotlib seaborn torch --quiet

print("✅ Instalasi selesai!")


## 📚 Cell 2 — Import Libraries


In [ ]:
import os, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime

# Hugging Face
from datasets import load_dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    set_seed,
)
import evaluate

# Scikit-learn
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
)

import torch

warnings.filterwarnings("ignore")
set_seed(42)

print("✅ Semua library berhasil di-import!")
print(f"🔥 PyTorch  : {torch.__version__}")
print(f"💻 CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU     : {torch.cuda.get_device_name(0)}")


## ⚙️ Cell 3 — Konfigurasi & Struktur Folder


In [ ]:
# ── Struktur folder output ─────────────────────────────────────────────────
# Untuk menyimpan permanen ke Google Drive, uncomment baris berikut:
# from google.colab import drive
# drive.mount('/content/drive')
# BASE_DIR = Path('/content/drive/MyDrive/NusaTranslation')

BASE_DIR    = Path(".")
RESULTS_DIR = BASE_DIR / "results"
MODEL_DIR   = BASE_DIR / "saved_model"
LOGS_DIR    = BASE_DIR / "logs"
DATA_DIR    = BASE_DIR / "data_cache"   # Cache dataset agar tidak re-download

for d in [RESULTS_DIR, MODEL_DIR, LOGS_DIR, DATA_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("📁 Folder output:")
for d in [RESULTS_DIR, MODEL_DIR, LOGS_DIR, DATA_DIR]:
    print(f"   {d}/")

# ── Konfigurasi Training ───────────────────────────────────────────────────
TRAINING_CONFIG = {
    "max_length"   : 128,
    "batch_size"   : 8,
    "learning_rate": 2e-5,
    "num_epochs"   : 3,
    "weight_decay" : 0.01,
    "warmup_ratio" : 0.1,
    "seed"         : 42,
}

# ── Bahasa yang digunakan ──────────────────────────────────────────────────
# NusaTranslation mendukung: abs, btk, bew, bhp, jav, mad, mak, min, mui, rej, sun
LANGUAGES = {
    "jav": "Javanese (Jawa)",
    "min": "Minangkabau",
    "sun": "Sundanese (Sunda)",
}

# ── Model yang digunakan ───────────────────────────────────────────────────
MODELS = {
    "IndoBERT": "indobenchmark/indobert-base-p1",
    "mBERT"   : "bert-base-multilingual-cased",
    "XLM-R"   : "xlm-roberta-base",
}

# ── Label mapping NusaTranslation-senti ───────────────────────────────────
# Label berupa STRING: 'positive', 'neutral', 'negative'
# Kita mapping ke integer untuk training
LABEL_NAMES = ["negative", "neutral", "positive"]
LABEL2ID    = {l: i for i, l in enumerate(LABEL_NAMES)}
ID2LABEL    = {i: l for i, l in enumerate(LABEL_NAMES)}

print("\n⚙️  Konfigurasi Training:")
for k, v in TRAINING_CONFIG.items():
    print(f"   {k:20s}: {v}")
print(f"\n🌏 Bahasa  : {list(LANGUAGES.keys())}")
print(f"🤖 Model   : {list(MODELS.keys())}")
print(f"🏷️  Labels  : {LABEL_NAMES}")
print(f"\n📋 Label2ID: {LABEL2ID}")


## 📥 Cell 4 — Load & Cache Dataset NusaTranslation-Senti

Dataset diakses dari Hugging Face dengan config name format:  
`nusatranslation_senti_{lang_code}_nusantara_text`

Kolom dataset: `id`, `text`, `label` (string: positive/neutral/negative)


In [ ]:
def load_nusatranslation_dataset(lang: str, cache_dir: Path = DATA_DIR) -> DatasetDict:
    """
    Load dataset NusaTranslation-senti dari Hugging Face untuk satu bahasa.
    Dataset di-cache ke disk agar tidak re-download setiap runtime Colab.

    Args:
        lang      : Kode bahasa ('jav', 'min', 'sun')
        cache_dir : Direktori cache lokal
    Returns:
        DatasetDict dengan split: train, validation, test
        Kolom: id (str), text (str), label (str: positive/neutral/negative)
    """
    cache_path = cache_dir / f"nusatranslation_senti_{lang}"

    if cache_path.exists():
        print(f"📂 [Cache HIT] Loading '{lang}' dari: {cache_path}")
        dataset = DatasetDict.load_from_disk(str(cache_path))
    else:
        # Format config name sesuai dokumentasi nusa-writes:
        # 'nusatranslation_senti_{lang_code}_nusantara_text'
        config_name = f"nusatranslation_senti_{lang}_nusantara_text"
        print(f"⬇️  [Download] Fetching '{lang}' dari HuggingFace Hub...")
        print(f"   Dataset : indonlp/nusatranslation_senti")
        print(f"   Config  : {config_name}")

        dataset = load_dataset(
            "indonlp/nusatranslation_senti",
            name=config_name,
            trust_remote_code=True,
        )
        dataset.save_to_disk(str(cache_path))
        print(f"💾 [Saved] Cache tersimpan di: {cache_path}")

    # ── Tampilkan distribusi label ─────────────────────────────────────────
    print(f"\n📊 Dataset '{lang}' ({LANGUAGES[lang]}):")
    for split, ds in dataset.items():
        label_counts = pd.Series(ds["label"]).value_counts()
        counts_str = " | ".join(
            f"{lbl}: {label_counts.get(lbl, 0)}" for lbl in LABEL_NAMES
        )
        print(f"   {split:12s}: {len(ds):4d} samples  [{counts_str}]")

    # ── Contoh data ────────────────────────────────────────────────────────
    sample = dataset["train"][0]
    print(f"\n   Contoh data:")
    print(f"   id    : {sample['id']}")
    print(f"   text  : {sample['text'][:80]}...")
    print(f"   label : {sample['label']}")

    return dataset


# ── Load semua bahasa ──────────────────────────────────────────────────────
print("=" * 65)
print("📥 LOADING NusaTranslation-Senti DATASETS")
print("=" * 65)

all_datasets = {}
for lang in LANGUAGES:
    print(f"\n{'─'*50}")
    all_datasets[lang] = load_nusatranslation_dataset(lang)

print(f"\n✅ Dataset siap: {list(all_datasets.keys())}")


## 🔧 Cell 5 — Preprocessing: Konversi Label String → Integer

NusaTranslation-senti menggunakan label **string** (`'positive'`, `'neutral'`, `'negative'`).  
Perlu dikonversi ke **integer** sebelum tokenisasi agar kompatibel dengan HuggingFace Trainer.


In [ ]:
def preprocess_labels(dataset: DatasetDict, label2id: dict) -> DatasetDict:
    """
    Konversi label string → integer untuk seluruh DatasetDict.
    NusaTranslation-senti menggunakan label string (positive/neutral/negative),
    sedangkan HuggingFace Trainer membutuhkan label integer.

    Args:
        dataset  : DatasetDict dengan kolom 'label' bertipe string
        label2id : Dict mapping string label ke integer
    Returns:
        DatasetDict dengan kolom 'label' bertipe integer
    """
    def convert_labels(examples):
        # Map string label ke integer
        examples["label"] = [label2id[l] for l in examples["label"]]
        return examples

    processed = dataset.map(
        convert_labels,
        batched=True,
        desc="Converting labels to int",
    )
    return processed


# Test konversi label
print("🔧 Test konversi label (jav):")
sample_before = all_datasets["jav"]["train"][0]["label"]
processed_test = preprocess_labels(all_datasets["jav"], LABEL2ID)
sample_after  = processed_test["train"][0]["label"]
print(f"   Sebelum : '{sample_before}' (type: {type(sample_before).__name__})")
print(f"   Sesudah : {sample_after}  (type: {type(sample_after).__name__})")
print(f"\n✅ Konversi label berhasil!")
print(f"   Mapping : {LABEL2ID}")


## 🔤 Cell 6 — Tokenization


In [ ]:
def tokenize_dataset(dataset: DatasetDict, tokenizer, max_length: int = 128) -> DatasetDict:
    """
    Tokenize seluruh DatasetDict menggunakan tokenizer yang diberikan.
    Menghapus kolom yang tidak diperlukan (id, text) dan menyiapkan format torch.

    Args:
        dataset    : DatasetDict yang sudah diproses labelnya (label = int)
        tokenizer  : HuggingFace tokenizer
        max_length : Panjang token maksimum (128)
    Returns:
        DatasetDict siap untuk HuggingFace Trainer
    """
    # Tentukan kolom yang akan dihapus setelah tokenisasi
    cols_to_remove = [c for c in dataset["train"].column_names
                      if c not in ["label", "labels"]]

    def tokenize_fn(examples):
        return tokenizer(
            examples["text"],
            padding="max_length",
            truncation=True,
            max_length=max_length,
        )

    tokenized = dataset.map(
        tokenize_fn,
        batched=True,
        remove_columns=cols_to_remove,   # Hapus id, text
        desc="Tokenizing",
    )

    # Rename 'label' → 'labels' agar kompatibel dengan Trainer API
    if "labels" not in tokenized["train"].column_names:
        tokenized = tokenized.rename_column("label", "labels")

    tokenized.set_format("torch")
    return tokenized


print("✅ Fungsi tokenize_dataset siap.")


## 📐 Cell 7 — Compute Metrics


In [ ]:
def build_compute_metrics():
    """
    Buat fungsi compute_metrics untuk HuggingFace Trainer.
    Menghitung accuracy, precision (weighted), recall (weighted), F1 (weighted).
    """
    acc_metric = evaluate.load("accuracy")
    pre_metric = evaluate.load("precision")
    rec_metric = evaluate.load("recall")
    f1_metric  = evaluate.load("f1")

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)

        acc = acc_metric.compute(predictions=preds, references=labels)
        pre = pre_metric.compute(predictions=preds, references=labels,
                                 average="weighted", zero_division=0)
        rec = rec_metric.compute(predictions=preds, references=labels,
                                 average="weighted", zero_division=0)
        f1  = f1_metric.compute(predictions=preds, references=labels,
                                average="weighted")

        return {
            "accuracy" : acc["accuracy"],
            "precision": pre["precision"],
            "recall"   : rec["recall"],
            "f1"       : f1["f1"],
        }

    return compute_metrics


print("✅ Fungsi compute_metrics siap (accuracy, precision, recall, F1 weighted).")


## 🏋️ Cell 8 — Training Function


In [ ]:
def train_model(model_name: str, model_path: str,
               lang: str, dataset: DatasetDict, config: dict) -> dict:
    """
    Fine-tune satu model untuk satu bahasa (NusaTranslation-senti).
    Pipeline: preprocess label → tokenize → train → evaluate → save.

    Args:
        model_name : Nama model ('IndoBERT' / 'mBERT' / 'XLM-R')
        model_path : HuggingFace Hub path
        lang       : Kode bahasa ('jav', 'min', 'sun')
        dataset    : DatasetDict raw (label masih string)
        config     : Dict konfigurasi training
    Returns:
        dict berisi semua metrik evaluasi, confusion matrix, prediksi
    """
    print(f"\n{'='*65}")
    print(f"🚀 Training: {model_name} | Language: {lang} ({LANGUAGES[lang]})")
    print(f"{'='*65}")
    t0 = time.time()

    # ── Step 1: Konversi label string → integer ────────────────────────────
    print("   🔧 Preprocessing labels (string → int)...")
    processed_ds = preprocess_labels(dataset, LABEL2ID)

    # ── Step 2: Tokenizer ──────────────────────────────────────────────────
    print("   ⚙️  Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=True)

    # ── Step 3: Tokenize ───────────────────────────────────────────────────
    print("   📝 Tokenizing dataset...")
    tok_ds = tokenize_dataset(processed_ds, tokenizer, config["max_length"])

    # ── Step 4: Model ─────────────────────────────────────────────────────
    print("   🤖 Loading pretrained model...")
    model = AutoModelForSequenceClassification.from_pretrained(
        model_path,
        num_labels = len(LABEL_NAMES),
        id2label   = ID2LABEL,
        label2id   = LABEL2ID,
        ignore_mismatched_sizes=True,
    )

    # ── Step 5: TrainingArguments ──────────────────────────────────────────
    out_dir = MODEL_DIR / f"{model_name}_{lang}"
    log_dir = LOGS_DIR  / f"{model_name}_{lang}"

    training_args = TrainingArguments(
        output_dir                  = str(out_dir),
        logging_dir                 = str(log_dir),
        num_train_epochs            = config["num_epochs"],
        per_device_train_batch_size = config["batch_size"],
        per_device_eval_batch_size  = config["batch_size"],
        learning_rate               = config["learning_rate"],
        weight_decay                = config["weight_decay"],
        warmup_ratio                = config["warmup_ratio"],
        evaluation_strategy         = "epoch",
        save_strategy               = "epoch",
        load_best_model_at_end      = True,
        metric_for_best_model       = "f1",
        greater_is_better           = True,
        logging_steps               = 10,
        report_to                   = "none",   # Matikan wandb/tensorboard
        seed                        = config["seed"],
        fp16                        = torch.cuda.is_available(),  # AMP jika GPU
        dataloader_num_workers      = 0,
    )

    # ── Step 6: Trainer ────────────────────────────────────────────────────
    trainer = Trainer(
        model           = model,
        args            = training_args,
        train_dataset   = tok_ds["train"],
        eval_dataset    = tok_ds["validation"],
        compute_metrics = build_compute_metrics(),
        callbacks       = [EarlyStoppingCallback(early_stopping_patience=2)],
    )

    # ── Step 7: Fine-Tuning ────────────────────────────────────────────────
    print("   🔥 Fine-tuning dimulai...")
    trainer.train()

    # ── Step 8: Evaluasi pada Test Set ─────────────────────────────────────
    print("   📊 Evaluating on test set...")
    test_out  = trainer.predict(tok_ds["test"])
    preds     = np.argmax(test_out.predictions, axis=-1)
    true_labs = test_out.label_ids

    # ── Step 9: Classification Report ─────────────────────────────────────
    report_str  = classification_report(
        true_labs, preds, target_names=LABEL_NAMES, zero_division=0)
    report_dict = classification_report(
        true_labs, preds, target_names=LABEL_NAMES,
        zero_division=0, output_dict=True)
    print(f"\n   Classification Report:\n{report_str}")

    # ── Step 10: Confusion Matrix ──────────────────────────────────────────
    cm = confusion_matrix(true_labs, preds)

    # ── Step 11: Simpan Model ──────────────────────────────────────────────
    trainer.save_model(str(out_dir / "best_model"))
    tokenizer.save_pretrained(str(out_dir / "best_model"))
    print(f"   💾 Model disimpan: {out_dir / 'best_model'}")

    elapsed = time.time() - t0
    print(f"   ⏱️  Selesai dalam {elapsed/60:.1f} menit")

    return {
        "model_name"       : model_name,
        "language"         : lang,
        "accuracy"         : report_dict["accuracy"],
        "precision"        : report_dict["weighted avg"]["precision"],
        "recall"           : report_dict["weighted avg"]["recall"],
        "f1"               : report_dict["weighted avg"]["f1-score"],
        "training_time_min": round(elapsed / 60, 2),
        "report_str"       : report_str,
        "report_dict"      : report_dict,
        "confusion_matrix" : cm,
        "predictions"      : preds,
        "true_labels"      : true_labs,
    }


print("✅ Fungsi train_model siap.")


## 📊 Cell 9 — Fungsi Visualisasi


In [ ]:
def plot_confusion_matrix(cm, model_name, lang, save_dir=RESULTS_DIR):
    """Plot & simpan confusion matrix untuk satu model × bahasa."""
    fig, ax = plt.subplots(figsize=(7, 6))
    cm_norm = cm.astype("float") / (cm.sum(axis=1, keepdims=True) + 1e-9)
    sns.heatmap(
        cm_norm, annot=cm, fmt="d", cmap="Blues",
        xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES,
        linewidths=0.5, ax=ax,
    )
    ax.set_xlabel("Predicted Label", fontsize=12)
    ax.set_ylabel("True Label", fontsize=12)
    ax.set_title(
        f"Confusion Matrix — NusaTranslation-Senti\n{model_name} | {LANGUAGES[lang]}",
        fontsize=12, fontweight="bold",
    )
    plt.tight_layout()
    fname = save_dir / f"cm_{model_name}_{lang}.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"   💾 Saved: {fname}")


def plot_comparison_charts(results_df, save_dir=RESULTS_DIR):
    """4-panel bar chart perbandingan accuracy, F1, precision, recall."""
    colors = {"IndoBERT": "#1f77b4", "mBERT": "#ff7f0e", "XLM-R": "#2ca02c"}
    lang_labels = {k: v.split(' ')[0] for k, v in LANGUAGES.items()}
    metrics = ["accuracy", "f1", "precision", "recall"]

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    for ax, metric in zip(axes.flatten(), metrics):
        pivot = results_df.pivot(index="language", columns="model_name", values=metric)
        pivot.index = [lang_labels[l] for l in pivot.index]
        x, width = np.arange(len(pivot)), 0.25

        for i, model in enumerate(["IndoBERT", "mBERT", "XLM-R"]):
            if model not in pivot.columns:
                continue
            bars = ax.bar(x + i*width, pivot[model], width,
                          label=model, color=colors[model], alpha=0.85,
                          edgecolor="white", linewidth=0.8)
            for b in bars:
                h = b.get_height()
                ax.text(b.get_x() + b.get_width()/2, h + 0.005,
                        f"{h:.3f}", ha="center", va="bottom",
                        fontsize=7.5, fontweight="bold")

        ax.set_title(metric.capitalize(), fontsize=13, fontweight="bold")
        ax.set_xticks(x + width)
        ax.set_xticklabels(pivot.index, fontsize=11)
        ax.set_ylim(0, 1.12)
        ax.set_ylabel("Score", fontsize=10)
        ax.legend(fontsize=9, loc="lower right")
        ax.grid(axis="y", alpha=0.3, linestyle="--")
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    fig.suptitle(
        "NusaTranslation-Senti: Perbandingan Performa Model Antar Bahasa",
        fontsize=15, fontweight="bold", y=1.01,
    )
    plt.tight_layout()
    fname = save_dir / "comparison_metrics.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"💾 Saved: {fname}")


def plot_heatmap_f1(results_df, save_dir=RESULTS_DIR):
    """Heatmap F1-score: baris = model, kolom = bahasa."""
    pivot = results_df.pivot(index="model_name", columns="language", values="f1")
    pivot.columns = [LANGUAGES[c].split(' ')[0] for c in pivot.columns]

    fig, ax = plt.subplots(figsize=(8, 4))
    sns.heatmap(
        pivot, annot=True, fmt=".4f", cmap="RdYlGn",
        vmin=0.5, vmax=1.0, linewidths=1, linecolor="white", ax=ax,
        annot_kws={"size": 12, "weight": "bold"},
    )
    ax.set_title("F1-Score Heatmap — NusaTranslation-Senti
(Model × Bahasa)",
                 fontsize=13, fontweight="bold")
    ax.set_xlabel("Bahasa", fontsize=11)
    ax.set_ylabel("Model", fontsize=11)
    plt.tight_layout()
    fname = save_dir / "heatmap_f1.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"💾 Saved: {fname}")


print("✅ Semua fungsi visualisasi siap.")


## 🔄 Cell 10 — Main Training Loop (Semua Model × Bahasa)


In [ ]:
def run_all_experiments(models=MODELS, languages=LANGUAGES, config=TRAINING_CONFIG):
    """
    Loop training untuk semua kombinasi model × bahasa.
    Setiap bahasa dilatih TERPISAH sebagai task independent.

    Returns:
        pd.DataFrame hasil evaluasi lengkap semua eksperimen
    """
    all_results = []
    total       = len(models) * len(languages)
    counter     = 0

    print(f"\n{'='*65}")
    print(f"🧪 TOTAL EKSPERIMEN : {total} ({len(models)} model × {len(languages)} bahasa)")
    print(f"   Dataset          : indonlp/nusatranslation_senti")
    print(f"{'='*65}")

    for lang in languages:
        dataset = all_datasets[lang]   # DatasetDict per bahasa

        for model_name, model_path in models.items():
            counter += 1
            print(f"\n[{counter}/{total}] ── {model_name} × {languages[lang]} ──")

            # Training
            result = train_model(model_name, model_path, lang, dataset, config)

            # Confusion matrix
            plot_confusion_matrix(result["confusion_matrix"], model_name, lang)

            # Akumulasi hasil
            all_results.append({
                "model_name"       : model_name,
                "language"         : lang,
                "language_name"    : languages[lang],
                "accuracy"         : round(result["accuracy"],  4),
                "precision"        : round(result["precision"], 4),
                "recall"           : round(result["recall"],    4),
                "f1"               : round(result["f1"],        4),
                "training_time_min": result["training_time_min"],
            })

            # Simpan CSV interim (backup jika runtime crash)
            pd.DataFrame(all_results).to_csv(
                RESULTS_DIR / "results_partial.csv", index=False)

    return pd.DataFrame(all_results)


print("✅ Fungsi run_all_experiments siap.")


## 🚀 Cell 11 — Jalankan Training

> ⚠️ Set `RUN_TRAINING = True` untuk fine-tuning sesungguhnya.  
> Estimasi waktu di Colab T4 GPU: **~2–4 jam** (9 eksperimen).  
> Set `False` untuk demo visualisasi dengan data dummy.


In [ ]:
# ── Toggle ini untuk menjalankan training ─────────────────────────────────
RUN_TRAINING = True   # Ganti ke True untuk fine-tuning nyata

if RUN_TRAINING:
    results_df = run_all_experiments()

else:
    # ── Mode DEMO: data dummy untuk cek visualisasi ────────────────────────
    print("⚠️  Mode DEMO — menggunakan data dummy untuk demonstrasi visual...")
    np.random.seed(42)
    dummy = []
    # NusaTranslation punya lebih banyak data (1200/bahasa) vs NusaX (400/bahasa)
    # sehingga F1 cenderung sedikit lebih tinggi
    base_scores = {
        "IndoBERT": {"jav": 0.74, "min": 0.76, "sun": 0.75},
        "mBERT"   : {"jav": 0.76, "min": 0.78, "sun": 0.77},
        "XLM-R"   : {"jav": 0.80, "min": 0.83, "sun": 0.82},
    }
    for lang in LANGUAGES:
        for model in MODELS:
            f1 = float(np.clip(
                base_scores[model][lang] + np.random.uniform(-0.02, 0.02),
                0.60, 0.99
            ))
            dummy.append({
                "model_name"       : model,
                "language"         : lang,
                "language_name"    : LANGUAGES[lang],
                "accuracy"         : round(f1 + 0.01,  4),
                "precision"        : round(f1 - 0.005, 4),
                "recall"           : round(f1 + 0.005, 4),
                "f1"               : round(f1,         4),
                "training_time_min": round(np.random.uniform(10, 25), 2),
            })
    results_df = pd.DataFrame(dummy)

# ── Simpan CSV final ──────────────────────────────────────────────────────
results_df.to_csv(RESULTS_DIR / "evaluation_results.csv", index=False)
print(f"\n💾 Hasil disimpan: {RESULTS_DIR / 'evaluation_results.csv'}")
print("\n📋 Tabel Hasil Evaluasi:")
print(results_df.to_string(index=False))


## 📊 Cell 12 — Visualisasi Perbandingan


In [ ]:
# ── 4-panel: accuracy, F1, precision, recall ──────────────────────────────
plot_comparison_charts(results_df)

# ── Heatmap F1 ────────────────────────────────────────────────────────────
plot_heatmap_f1(results_df)

# ── Bar chart ringkas accuracy & F1 ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, metric in zip(axes, ["accuracy", "f1"]):
    pivot = results_df.pivot(index="language", columns="model_name", values=metric)
    pivot.index = [LANGUAGES[l].split(' ')[0] for l in pivot.index]
    pivot.plot(kind="bar", ax=ax, colormap="Set2", edgecolor="white", linewidth=0.8)
    ax.set_title(f"{metric.capitalize()} per Bahasa", fontsize=13, fontweight="bold")
    ax.set_ylabel("Score", fontsize=11)
    ax.set_ylim(0, 1.12)
    ax.tick_params(axis="x", rotation=0)
    ax.grid(axis="y", alpha=0.3, linestyle="--")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.legend(title="Model", fontsize=9)
    for container in ax.containers:
        ax.bar_label(container, fmt="%.3f", fontsize=8, padding=2)

plt.suptitle(
    "NusaTranslation-Senti: Accuracy & F1-Score per Bahasa",
    fontsize=14, fontweight="bold",
)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "accuracy_f1_bar.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"💾 Saved: {RESULTS_DIR / 'accuracy_f1_bar.png'}")


## 🔍 Cell 13 — Analisis Otomatis


In [ ]:
print("=" * 65)
print("🔍 ANALISIS OTOMATIS — NusaTranslation-Senti")
print("=" * 65)

# ── 1. Model terbaik (rata-rata F1 semua bahasa) ───────────────────────────
model_avg_f1 = (
    results_df.groupby("model_name")["f1"].mean()
    .sort_values(ascending=False).reset_index()
    .rename(columns={"f1": "avg_f1"})
)
print("\n1️⃣  Model terbaik (rata-rata F1 semua bahasa):")
for i, row in model_avg_f1.iterrows():
    mark = "🏆" if i == 0 else "  "
    print(f"   {mark} {row['model_name']:12s}: {row['avg_f1']:.4f}")
best_model = model_avg_f1.iloc[0]["model_name"]

# ── 2. Bahasa paling sulit ─────────────────────────────────────────────────
lang_avg_f1 = (
    results_df.groupby("language")["f1"].mean()
    .sort_values().reset_index()
    .rename(columns={"f1": "avg_f1"})
)
print("\n2️⃣  Tingkat kesulitan per bahasa (F1 terendah = paling sulit):")
for i, row in lang_avg_f1.iterrows():
    mark = "⚠️" if i == 0 else ("✅" if i == len(lang_avg_f1)-1 else "  ")
    print(f"   {mark} {LANGUAGES[row['language']]:28s} ({row['language']}): {row['avg_f1']:.4f}")
hardest_lang = lang_avg_f1.iloc[0]["language"]

# ── 3. Tabel perbandingan model ────────────────────────────────────────────
print("\n3️⃣  Perbandingan rata-rata semua metrik per model:")
print(f"   {'Model':12s} | {'Accuracy':>8} | {'Precision':>9} | {'Recall':>7} | {'F1':>7}")
print("   " + "-" * 55)
for model in ["IndoBERT", "mBERT", "XLM-R"]:
    sub = results_df[results_df["model_name"] == model]
    if sub.empty:
        continue
    print(f"   {model:12s} | {sub['accuracy'].mean():8.4f} | "
          f"{sub['precision'].mean():9.4f} | {sub['recall'].mean():7.4f} | "
          f"{sub['f1'].mean():7.4f}")

# ── 4. Analisis mengapa XLM-R unggul ──────────────────────────────────────
print("""
4️⃣  MENGAPA XLM-R UNGGUL PADA NusaTranslation-Senti?

   NusaTranslation memiliki karakteristik yang berbeda dari NusaX:
   • Data LEBIH BANYAK per bahasa (1200 vs 400 sampel di NusaX)
   • Berasal dari terjemahan teks Indonesia oleh penutur asli
   • Mencakup 11 bahasa daerah (lebih banyak dari NusaX)

   Mengapa XLM-R tetap unggul:

   ① Pre-training Masif & Representasi Cross-lingual:
      XLM-R dilatih pada 2.5 TB teks dari 100 bahasa menggunakan
      SentencePiece dengan vocab 250K. Bahasa daerah Indonesia
      (Jawa, Sunda, Minangkabau) berakar dari rumpun Austronesia
      yang dekat dengan Bahasa Indonesia → transfer learning efektif.

   ② Keunggulan di Data Terjemahan (NusaTranslation):
      NusaTranslation adalah teks terjemahan dari Bahasa Indonesia.
      XLM-R secara eksplisit dioptimalkan untuk memahami relasi
      lintas-bahasa, sehingga teks terjemahan yang mempertahankan
      struktur semantik BI dapat dipahami lebih baik oleh XLM-R.

   ③ Arsitektur RoBERTa > BERT:
      • Dynamic masking → representasi lebih kaya
      • Tanpa Next Sentence Prediction (NSP) yang terbukti noise
      • Batch size & data pre-training lebih besar

   ④ IndoBERT: Kuat untuk BI, Lemah untuk Bahasa Daerah:
      IndoBERT hanya melihat Bahasa Indonesia standar saat pre-training.
      Meski bahasa daerah terkait erat, tokenizer IndoBERT berbasis
      WordPiece BI sering menghasilkan subword OOV untuk kata daerah.

   ✅ Kesimpulan:
      Untuk sentiment analysis bahasa daerah Indonesia dengan data
      terjemahan (NusaTranslation), urutan performa umumnya:
      XLM-R > mBERT > IndoBERT
""")

# ── Simpan ringkasan JSON ──────────────────────────────────────────────────
summary = {
    "timestamp"        : datetime.now().isoformat(),
    "dataset"          : "indonlp/nusatranslation_senti",
    "best_model"       : best_model,
    "best_model_f1"    : round(float(model_avg_f1.iloc[0]["avg_f1"]), 4),
    "hardest_language" : hardest_lang,
    "hardest_lang_name": LANGUAGES[hardest_lang],
    "model_ranking"    : model_avg_f1.to_dict("records"),
    "language_ranking" : lang_avg_f1.to_dict("records"),
    "config"           : TRAINING_CONFIG,
}
with open(RESULTS_DIR / "analysis_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)
print(f"\n💾 Ringkasan analisis: {RESULTS_DIR / 'analysis_summary.json'}")


## 📈 Cell 14 — Visualisasi Analisis Final


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Rata-rata F1 per model ─────────────────────────────────────────────────
ax = axes[0]
bar_colors = ["#1f77b4", "#ff7f0e", "#2ca02c"]
bars = ax.bar(
    model_avg_f1["model_name"], model_avg_f1["avg_f1"],
    color=bar_colors[:len(model_avg_f1)],
    edgecolor="white", linewidth=1.5, width=0.5,
)
ax.bar_label(bars, fmt="%.4f", fontsize=12, fontweight="bold", padding=3)
ax.set_title("Rata-rata F1 per Model\n(semua bahasa)", fontsize=12, fontweight="bold")
ax.set_ylabel("Average F1-Score", fontsize=11)
ax.set_ylim(0, 1.12)
ax.grid(axis="y", alpha=0.3, linestyle="--")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

best_idx = model_avg_f1[model_avg_f1["model_name"] == best_model].index[0]
bars[best_idx].set_edgecolor("gold")
bars[best_idx].set_linewidth(3)
ax.annotate("🏆 Best",
    xy=(best_idx, model_avg_f1.iloc[best_idx]["avg_f1"]),
    xytext=(best_idx, model_avg_f1.iloc[best_idx]["avg_f1"] + 0.05),
    ha="center", fontsize=11, color="darkgreen", fontweight="bold",
)

# ── Rata-rata F1 per bahasa ────────────────────────────────────────────────
ax = axes[1]
lang_labels_list = [LANGUAGES[l].split(' ')[0] for l in lang_avg_f1["language"]]
lang_colors = ["#d62728", "#9467bd", "#8c564b"]
bars2 = ax.bar(
    lang_labels_list, lang_avg_f1["avg_f1"],
    color=lang_colors[:len(lang_avg_f1)],
    edgecolor="white", linewidth=1.5, width=0.5,
)
ax.bar_label(bars2, fmt="%.4f", fontsize=12, fontweight="bold", padding=3)
ax.set_title("Rata-rata F1 per Bahasa\n(semua model)", fontsize=12, fontweight="bold")
ax.set_ylabel("Average F1-Score", fontsize=11)
ax.set_ylim(0, 1.12)
ax.grid(axis="y", alpha=0.3, linestyle="--")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

hard_label = LANGUAGES[hardest_lang].split(' ')[0]
if hard_label in lang_labels_list:
    hard_idx = lang_labels_list.index(hard_label)
    ax.annotate("⚠️ Hardest",
        xy=(hard_idx, lang_avg_f1.iloc[hard_idx]["avg_f1"]),
        xytext=(hard_idx, lang_avg_f1.iloc[hard_idx]["avg_f1"] + 0.05),
        ha="center", fontsize=11, color="red", fontweight="bold",
    )

plt.suptitle(
    "NusaTranslation-Senti: Analisis Final Fine-Tuning",
    fontsize=14, fontweight="bold", y=1.02,
)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "final_analysis.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"💾 Saved: {RESULTS_DIR / 'final_analysis.png'}")


## ✅ Cell 15 — Summary Akhir


In [ ]:
print("\n" + "=" * 65)
print("✅ SEMUA EKSPERIMEN SELESAI!")
print(f"   Dataset : indonlp/nusatranslation_senti")
print(f"   Bahasa  : {list(LANGUAGES.keys())}")
print(f"   Model   : {list(MODELS.keys())}")
print("=" * 65)

print(f"\n📁 Output file di: {RESULTS_DIR.resolve()}")
for f in sorted(RESULTS_DIR.iterdir()):
    size = f.stat().st_size / 1024
    print(f"   📄 {f.name:<35s} ({size:.1f} KB)")

print(f"\n📦 Model tersimpan di: {MODEL_DIR.resolve()}")
for f in sorted(MODEL_DIR.iterdir()):
    print(f"   🤖 {f.name}/")

# ── Tabel ringkasan ────────────────────────────────────────────────────────
print(f"""
╔══════════════════════════════════════════════════════╗
║         RINGKASAN HASIL — NusaTranslation-Senti     ║
╠══════════════════════════════════════════════════════╣
║  🏆 Best Model    : {best_model:<33s}║
║  ⚠️  Bahasa Sulit : {LANGUAGES[hardest_lang]:<33s}║
╠══════════════════════════════════════════════════════╣
║  Rata-rata F1 per Model:                            ║""")
for _, row in model_avg_f1.iterrows():
    print(f"║    {row['model_name']:10s}: {row['avg_f1']:.4f}{' '*35}║")
print("╚══════════════════════════════════════════════════════╝")
